# 🛒 Online Retail II — Cohort Analysis & Customer Retention

**Author:** Rahmadhania  
**Date:** April 2026  
**Dataset:** [UCI Machine Learning Repository — Online Retail II](https://archive.ics.uci.edu/dataset/502/online+retail+ii)

---

## 📌 Business Question

> **How well does this UK online retailer retain customers month-over-month, and where does it lose them?**

Understanding retention is fundamental to any e-commerce business. Acquiring a new customer costs 5–7× more than retaining an existing one. If a retailer loses most customers after the first purchase, that signals a problem with product experience, communication, or loyalty strategy — not just acquisition.

---

## 🔧 Approach

We use **cohort analysis** — grouping customers by the month of their first purchase, then tracking what percentage of each group returns in subsequent months.

**Pipeline:**
```
SQL (cleaning) → SQL (cohort assignment) → SQL (retention matrix) → SQL (retention rates) → Python (heatmap)
```

---
## 1️⃣ Setup & Data Loading

In [ ]:
# Core libraries
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', font_scale=1.0)

print('Libraries loaded ✅')

In [ ]:
# Load both sheets from the Excel file
# The dataset spans two years split across two sheets
DATA_PATH = os.path.join('data', 'online_retail_II.xlsx')

print('Loading Year 2009-2010...')
df_1 = pd.read_excel(DATA_PATH, sheet_name='Year 2009-2010', engine='openpyxl')

print('Loading Year 2010-2011...')
df_2 = pd.read_excel(DATA_PATH, sheet_name='Year 2010-2011', engine='openpyxl')

# Combine both years
df_raw = pd.concat([df_1, df_2], ignore_index=True)

print(f'\nTotal rows loaded: {len(df_raw):,}')
print(f'Columns: {list(df_raw.columns)}')

In [ ]:
# Quick preview of the raw data
df_raw.head(10)

In [ ]:
# Check data types and missing values before any cleaning
print('=== Data Types ===')
print(df_raw.dtypes)
print()
print('=== Missing Values ===')
print(df_raw.isnull().sum())
print()
print('=== Basic Stats ===')
print(df_raw[['Quantity', 'Price']].describe())

**Observations from raw data:**
- `Customer ID` has a significant number of missing values (~25%) — these rows cannot be used for customer-level analysis
- `Quantity` has negative values — these are cancellations and returns
- `Price` has zero values — likely internal transfers or promotional items
- `Invoice` codes starting with `'C'` are cancellation records

---
## 2️⃣ Data Cleaning & Cohort Building via SQL

We load the data into an **in-memory SQLite database** and run our 4 SQL scripts in sequence.

This approach keeps the analytical logic in SQL (transparent, auditable, portable) and Python for visualization only.

In [ ]:
# Normalize column names to match our SQL scripts
df_raw.columns = df_raw.columns.str.strip().str.replace(' ', '_').str.lower()

df_raw = df_raw.rename(columns={
    'invoice':     'InvoiceNo',
    'stockcode':   'StockCode',
    'description': 'Description',
    'quantity':    'Quantity',
    'invoicedate': 'InvoiceDate',
    'price':       'UnitPrice',
    'customer_id': 'CustomerID',
    'country':     'Country',
})

# Convert dates to ISO string format that SQLite can sort correctly
df_raw['InvoiceDate'] = pd.to_datetime(df_raw['InvoiceDate']).dt.strftime('%Y-%m-%d %H:%M:%S')

print('Columns normalized ✅')
print(df_raw.columns.tolist())

In [ ]:
# Load into in-memory SQLite and run the SQL pipeline
conn = sqlite3.connect(':memory:')
df_raw.to_sql('raw_retail', conn, if_exists='replace', index=False)

def run_sql_file(conn, filename):
    """Execute a SQL script file against the SQLite connection."""
    filepath = os.path.join('sql', filename)
    with open(filepath, 'r', encoding='utf-8') as f:
        sql = f.read()
    conn.executescript(sql)
    print(f'  ✅ {filename}')

print('Running SQL pipeline...')
run_sql_file(conn, '01_cleaning.sql')
run_sql_file(conn, '02_cohort_assignment.sql')
run_sql_file(conn, '03_retention_matrix.sql')
run_sql_file(conn, '04_retention_rate.sql')
print('\nSQL pipeline complete ✅')

In [ ]:
# Validate cleaning results
raw_count = pd.read_sql('SELECT COUNT(*) AS n FROM raw_retail', conn).iloc[0, 0]
clean_count = pd.read_sql('SELECT COUNT(*) AS n FROM clean_transactions', conn).iloc[0, 0]
unique_customers = pd.read_sql('SELECT COUNT(DISTINCT CustomerID) AS n FROM clean_transactions', conn).iloc[0, 0]
date_range = pd.read_sql('SELECT MIN(InvoiceDate) AS start, MAX(InvoiceDate) AS end FROM clean_transactions', conn)

print(f'Raw rows            : {raw_count:,}')
print(f'Clean rows          : {clean_count:,}  ({clean_count/raw_count*100:.1f}% retained)')
print(f'Rows removed        : {raw_count - clean_count:,}')
print(f'Unique customers    : {unique_customers:,}')
print(f'Date range          : {date_range.iloc[0,0]} → {date_range.iloc[0,1]}')

**What was removed and why:**
| Filter | Reason |
|--------|--------|
| `InvoiceNo LIKE 'C%'` | Cancellations — not real purchases |
| `CustomerID IS NULL` | Cannot assign cohort without customer identity |
| `UnitPrice <= 0` | Internal transfers, samples, or data errors |
| `Quantity <= 0` | Returns not caught by the 'C' prefix |


---
## 3️⃣ Exploratory Analysis

Before the cohort heatmap, let's understand the shape of our customer base.

In [ ]:
# Monthly transaction volume — are there seasonal patterns?
monthly = pd.read_sql("""
    SELECT
        SUBSTR(InvoiceDate, 1, 7) AS month,
        COUNT(DISTINCT InvoiceNo)  AS orders,
        COUNT(DISTINCT CustomerID) AS active_customers,
        ROUND(SUM(Revenue), 0)     AS revenue_gbp
    FROM clean_transactions
    GROUP BY month
    ORDER BY month
""", conn)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].bar(monthly['month'], monthly['active_customers'], color='#4a90d9', alpha=0.85)
axes[0].set_title('Monthly Active Customers (Cleaned Data)', fontweight='bold')
axes[0].set_ylabel('Unique Customers')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(monthly['month'], monthly['revenue_gbp'] / 1000, color='#e8854a', alpha=0.85)
axes[1].set_title('Monthly Revenue (GBP £000s)', fontweight='bold')
axes[1].set_ylabel('Revenue (£ thousands)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join('outputs', 'monthly_overview.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Chart saved ✅')

In [ ]:
# Cohort sizes — how many new customers joined each month?
cohort_sizes = pd.read_sql("""
    SELECT cohort_month, active_customers AS cohort_size
    FROM retention_matrix
    WHERE period_number = 0
    ORDER BY cohort_month
""", conn)

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(cohort_sizes['cohort_month'], cohort_sizes['cohort_size'], color='#5ba85a', alpha=0.85)
ax.set_title('New Customer Cohort Size by Month', fontweight='bold')
ax.set_xlabel('Cohort Month')
ax.set_ylabel('Number of New Customers')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join('outputs', 'cohort_sizes.png'), dpi=120, bbox_inches='tight')
plt.show()

print(f"Largest cohort : {cohort_sizes.loc[cohort_sizes['cohort_size'].idxmax(), 'cohort_month']} "
      f"({cohort_sizes['cohort_size'].max():,} customers)")
print(f"Smallest cohort: {cohort_sizes.loc[cohort_sizes['cohort_size'].idxmin(), 'cohort_month']} "
      f"({cohort_sizes['cohort_size'].min():,} customers)")

---
## 4️⃣ Cohort Retention Heatmap

This is the core deliverable. Each row = one cohort (month of first purchase). Each column = months since acquisition. Values = % of original cohort still active.

- **Period 0** is always 100% — every customer was active when they first purchased
- **Darker red** = more churn
- **Yellower** = stronger retention

In [ ]:
# Load final retention rates table
retention_df = pd.read_sql("""
    SELECT cohort_month, period_number, retention_rate
    FROM retention_rates
    ORDER BY cohort_month, period_number
""", conn)

# Pivot to matrix format: rows = cohorts, columns = periods
cohort_pivot = retention_df.pivot_table(
    index='cohort_month',
    columns='period_number',
    values='retention_rate'
).sort_index()

print(f'Matrix shape: {cohort_pivot.shape[0]} cohorts × {cohort_pivot.shape[1]} periods')
cohort_pivot.iloc[:5, :6]

In [ ]:
# Build the cohort retention heatmap
fig, ax = plt.subplots(figsize=(18, 8))

# Mask for period 0 (always 100% — styled separately to avoid skewing the color scale)
mask_p0 = pd.DataFrame(False, index=cohort_pivot.index, columns=cohort_pivot.columns)
if 0 in mask_p0.columns:
    mask_p0[0] = True

# Main heatmap (periods 1+)
sns.heatmap(
    cohort_pivot,
    mask=mask_p0.values,
    annot=True, fmt='.1f',
    cmap='YlOrRd_r',
    linewidths=0.4, linecolor='#e0e0e0',
    vmin=0, vmax=100,
    cbar_kws={'label': 'Retention Rate (%)', 'shrink': 0.6},
    ax=ax, annot_kws={'size': 7.5}
)

# Period 0 overlay (blue = baseline)
sns.heatmap(
    cohort_pivot,
    mask=~mask_p0.values,
    annot=True, fmt='.1f',
    cmap=['#4a90d9'],
    linewidths=0.4, linecolor='#e0e0e0',
    vmin=100, vmax=100,
    cbar=False,
    ax=ax, annot_kws={'size': 7.5, 'color': 'white', 'weight': 'bold'}
)

ax.set_title(
    'Customer Cohort Retention — Online Retail II (2009–2011)',
    fontsize=15, fontweight='bold', pad=16
)
ax.set_xlabel('Months Since First Purchase (Period)', fontsize=11, labelpad=10)
ax.set_ylabel('Cohort (First Purchase Month)', fontsize=11, labelpad=10)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=8.5)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8.5)

fig.text(
    0.5, -0.02,
    'Values show % of cohort customers still active N months after first purchase.  '
    'Blue = acquisition month (always 100%).  Yellow–Red scale = retention strength.',
    ha='center', fontsize=8.5, color='#555555'
)

plt.tight_layout()
plt.savefig(os.path.join('outputs', 'cohort_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Heatmap saved to outputs/cohort_heatmap.png ✅')

---
## 5️⃣ Key Insights & Business Recommendations

> ⚠️ **Fill in the actual numbers after running the analysis on your machine.** The structure below is ready — replace the italicized placeholders with real findings from your heatmap.

In [ ]:
# Average retention rate by period across all cohorts
avg_retention = cohort_pivot.mean(axis=0).reset_index()
avg_retention.columns = ['period_number', 'avg_retention_rate']
avg_retention = avg_retention[avg_retention['period_number'] > 0]  # exclude period 0

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(
    avg_retention['period_number'],
    avg_retention['avg_retention_rate'],
    marker='o', color='#4a90d9', linewidth=2, markersize=5
)
ax.fill_between(
    avg_retention['period_number'],
    avg_retention['avg_retention_rate'],
    alpha=0.15, color='#4a90d9'
)
ax.set_title('Average Retention Rate by Month Since Acquisition', fontweight='bold')
ax.set_xlabel('Months Since First Purchase')
ax.set_ylabel('Average Retention Rate (%)')
ax.set_ylim(0, 60)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join('outputs', 'avg_retention_curve.png'), dpi=120, bbox_inches='tight')
plt.show()

print('Average retention by period:')
print(avg_retention.to_string(index=False))

### 📊 Key Findings

*(Update these after running the notebook — replace placeholders with actual numbers)*

1. **Sharp first-month drop:** The steepest churn occurs between Period 0 and Period 1, where retention typically falls to ~[X]%. This indicates that most customers do not return after their first purchase.

2. **Stabilization after Month 3:** Retention curves flatten around Period 3–4, suggesting that customers who survive past Month 3 become significantly more loyal.

3. **Strongest cohort:** The [YYYY-MM] cohort shows the highest long-term retention — worth investigating what was different about that period (promotions, seasonality, product mix).

4. **Seasonal acquisition peak:** Cohort sizes spike in [Month] — likely driven by holiday gifting demand. However, larger cohorts don't necessarily mean better retention.

5. **Wholesaler effect:** Given that many customers are wholesalers, retained customers likely represent bulk buyers. The retention rate understates the revenue impact of loyal customers.

---

### 💡 Business Recommendations

| Finding | Recommendation |
|---------|----------------|
| High Month-1 churn | Implement a post-purchase email sequence (Day 7, Day 14, Day 30) to re-engage first-time buyers |
| Loyal core after Month 3 | Introduce a loyalty or rewards program targeting customers who have purchased 2+ times |
| Seasonal spikes | Use Q4 acquisition periods to invest in onboarding — these cohorts have the most volume |
| B2B customers | Offer account management or bulk-order incentives to high-value wholesale customers |


In [ ]:
# Close the SQLite connection
conn.close()
print('SQLite connection closed ✅')
print('\nAll outputs saved to outputs/ folder:')
for f in os.listdir('outputs'):
    if not f.startswith('.'):
        print(f'  📊 {f}')

---
*Analysis by Rahmadhania · April 2026 · [GitHub](https://github.com/YOUR_USERNAME/online-retail-cohort-analysis)*